# Integridad y preprocesamiento

Cristhian Pazmino Parra · Universidad Espíritu Santo

Verifica los bytes y la separación del manifiesto, sin crear aprobaciones de revisión profesional.

Referente: anotaciones publicadas sin adjudicación local. Los datos permanecen fuera del repositorio.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
assert (ROOT / "src").is_dir(), "Ejecutar desde la raíz del repositorio"
sys.path.insert(0, str(ROOT))
LOCAL = Path.home() / "AppData/Local/DentalAI"
DATA = LOCAL / "data/research_v2"
REPORTS = ROOT / "reports/completion"
def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))
from IPython.display import display, Markdown, Image


In [2]:
from src.data.research import verify_research_contract, prepare_image, REFERENCE
contract, rows = verify_research_contract(ROOT / "configs/data/contracts/research_v2.json", for_training=True)
assert contract["reference_status"] == REFERENCE
print("Contrato de investigación verificado; no acredita aprobación clínica")

Contrato de investigación verificado; no acredita aprobación clínica


In [3]:
from collections import defaultdict
groups = defaultdict(set)
for row in rows:
    groups[row["group_id"]].add(row["split"])
assert all(len(v) == 1 for v in groups.values())
print(len(groups), "grupos de similitud; identidad de paciente desconocida")

1062 grupos de similitud; identidad de paciente desconocida


In [4]:
import cv2, numpy as np
row = next(r for r in rows if r["split"] == "val")
raw = DATA.parent / "raw_verified" / row["source_image"]
im = cv2.imdecode(np.frombuffer(raw.read_bytes(), dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
derived = cv2.imread(str(DATA / row["image"]), cv2.IMREAD_GRAYSCALE)
assert np.array_equal(prepare_image(im), derived)
print("Derivado idéntico; forma:", derived.shape)

Derivado idéntico; forma: (668, 1280)


Reconstrucción desde cero: `src.data.research.build_research_view`. Una nueva política necesita una nueva vista y contrato; el ejecutor rechaza sobrescrituras.